# 🌍 Global Economic Digital Twin — Exploratory Data Analysis

**Author:** Yash Chaudhary | [github.com/yash752-stack](https://github.com/yash752-stack)  
**Project:** [econ-twin](https://github.com/yash752-stack/econ-twin)

This notebook explores the macro dataset produced by `pipeline/data_pipeline.py`.

**Contents:**
1. Dataset overview & schema
2. Global GDP growth trends
3. Inflation analysis
4. Trade balance & current account
5. Commodity price correlations
6. Feature correlation heatmap
7. Country clustering (K-Means)
8. Recession signal analysis
9. XGBoost feature importance preview

> **Prerequisites:** Run `python run_all.py` first to generate `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import warnings
warnings.filterwarnings('ignore')

# Notebook-wide style
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Libraries loaded ✅')

---
## 1. Dataset Overview

In [ ]:
econ_df   = pd.read_csv('../data/processed/economic_data.csv')
trade_df  = pd.read_csv('../data/processed/trade_data.csv')
comm_df   = pd.read_csv('../data/processed/commodity_data.csv')
net_df    = pd.read_csv('../data/processed/network_metrics.csv')
shock_df  = pd.read_csv('../data/processed/shock_results.csv')
rec_df    = pd.read_csv('../data/processed/recession_probabilities.csv')
fc_df     = pd.read_csv('../data/processed/ml_forecasts.csv')

with open('../data/processed/model_metrics.json') as f:
    model_metrics = json.load(f)

print('=== Dataset Summary ===')
print(f'Economic data : {econ_df.shape[0]:,} rows × {econ_df.shape[1]} columns')
print(f'Trade flows   : {trade_df.shape[0]:,} rows × {trade_df.shape[1]} columns')
print(f'Commodity data: {comm_df.shape[0]} rows × {comm_df.shape[1]} columns')
print(f'Countries     : {econ_df["iso3"].nunique()}')
print(f'Year range    : {econ_df["year"].min()} – {econ_df["year"].max()}')
print(f'Regions       : {sorted(econ_df["region"].unique())}')

In [ ]:
econ_df.describe().round(2)

In [ ]:
# Missing value audit
missing = econ_df.isnull().sum()
missing_pct = (missing / len(econ_df) * 100).round(2)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(audit[audit['Missing Count'] > 0])
print('\nNo missing values ✅' if missing.sum() == 0 else f'⚠️  {missing.sum()} missing values found')

---
## 2. Global GDP Growth Trends

In [ ]:
# World-average GDP growth per year
world_avg = econ_df.groupby('year')['gdp_growth_pct'].mean().reset_index()
world_avg.columns = ['year', 'world_avg_growth']

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(world_avg['year'], world_avg['world_avg_growth'],
                alpha=0.3, color='#4caf8a')
ax.plot(world_avg['year'], world_avg['world_avg_growth'],
        color='#4caf8a', linewidth=2.5, marker='o')
ax.axhline(y=0, color='white', linestyle='--', alpha=0.4)

# Annotate key events
events = {
    2012: 'Eurozone\nCrisis',
    2015: 'Oil Crash\nChina Slow',
    2020: 'COVID-19',
    2021: 'Rebound',
    2022: 'Energy\nCrisis',
}
for year, label in events.items():
    y_val = world_avg[world_avg['year'] == year]['world_avg_growth'].values[0]
    ax.annotate(label, xy=(year, y_val), xytext=(year, y_val + 1.2),
                ha='center', fontsize=8, color='#aaaaaa',
                arrowprops=dict(arrowstyle='->', color='#666666', lw=0.8))

ax.set_title('World Average GDP Growth 2010–2023', fontsize=14, pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('GDP Growth %')
ax.set_xticks(world_avg['year'])
plt.tight_layout()
plt.savefig('../data/processed/eda_gdp_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# G7 vs Emerging Markets comparison
G7 = ['USA', 'GBR', 'DEU', 'FRA', 'JPN', 'CAN', 'ITA']
EM = ['CHN', 'IND', 'BRA', 'RUS', 'MYS', 'IDN', 'THA', 'EGY']

g7_avg = econ_df[econ_df['iso3'].isin(G7)].groupby('year')['gdp_growth_pct'].mean()
em_avg = econ_df[econ_df['iso3'].isin(EM)].groupby('year')['gdp_growth_pct'].mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=g7_avg.index, y=g7_avg.values, name='G7',
                          line=dict(color='#4a7fb5', width=2), mode='lines+markers'))
fig.add_trace(go.Scatter(x=em_avg.index, y=em_avg.values, name='Emerging Markets',
                          line=dict(color='#f0a050', width=2), mode='lines+markers'))
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(title='G7 vs Emerging Markets: GDP Growth Gap',
                   xaxis_title='Year', yaxis_title='GDP Growth %',
                   height=400, template='plotly_dark')
fig.show()

---
## 3. Inflation Analysis

In [ ]:
# Distribution of inflation across all countries × years
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(econ_df['inflation_pct'].clip(-2, 30), bins=50,
              color='#e05c5c', alpha=0.8, edgecolor='none')
axes[0].axvline(x=2, color='white', linestyle='--', alpha=0.6, label='2% target')
axes[0].set_title('Inflation Distribution (all countries × years)')
axes[0].set_xlabel('Inflation %')
axes[0].legend()

# Inflation by region — 2022
inf_region = econ_df[econ_df['year'] == 2022].groupby('region')['inflation_pct'].mean().sort_values()
axes[1].barh(inf_region.index, inf_region.values,
              color=['#e05c5c' if v > 5 else '#4caf8a' for v in inf_region.values])
axes[1].axvline(x=2, color='white', linestyle='--', alpha=0.5)
axes[1].set_title('Average Inflation by Region (2022)')
axes[1].set_xlabel('Inflation %')

plt.tight_layout()
plt.show()

In [ ]:
# Inflation heatmap — highlight 2020-2022 surge
pivot_inf = econ_df.pivot_table(index='country', columns='year', values='inflation_pct')

fig = go.Figure(go.Heatmap(
    z=pivot_inf.values,
    x=pivot_inf.columns.tolist(),
    y=pivot_inf.index.tolist(),
    colorscale='Reds', zmin=0, zmax=20,
    colorbar=dict(title='Inflation %'),
))
fig.update_layout(title='Inflation Heatmap — All Countries × Years',
                   height=700, template='plotly_dark')
fig.show()

---
## 4. Trade Balance Analysis

In [ ]:
# Trade surplus / deficit — 2022 snapshot
latest = econ_df[econ_df['year'] == 2022].copy()
latest['trade_balance_bn'] = (latest['exports_usd'] - latest['imports_usd']) / 1e9
latest_sorted = latest.sort_values('trade_balance_bn')

fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#e05c5c' if v < 0 else '#4caf8a' for v in latest_sorted['trade_balance_bn']]
ax.barh(latest_sorted['country'], latest_sorted['trade_balance_bn'], color=colors)
ax.axvline(x=0, color='white', linewidth=0.8)
ax.set_xlabel('Trade Balance ($B)')
ax.set_title('Trade Balance by Country — 2022 (green = surplus, red = deficit)')
plt.tight_layout()
plt.show()

print('\nTop 5 Surplus Countries:')
print(latest.nlargest(5, 'trade_balance_bn')[['country', 'trade_balance_bn']].to_string(index=False))
print('\nTop 5 Deficit Countries:')
print(latest.nsmallest(5, 'trade_balance_bn')[['country', 'trade_balance_bn']].to_string(index=False))

---
## 5. Commodity Price Correlations

In [ ]:
# Commodity price trends
comm_norm = comm_df.copy()
for col in ['oil_brent', 'gold', 'copper', 'wheat', 'natural_gas']:
    comm_norm[col] = comm_norm[col] / comm_norm[col].iloc[0] * 100  # index to 2010=100

fig = go.Figure()
colors = {'oil_brent': '#f0a050', 'gold': '#FFD700', 'copper': '#CD7F32',
           'wheat': '#8BC34A', 'natural_gas': '#4fc3f7'}

for comm, color in colors.items():
    fig.add_trace(go.Scatter(x=comm_df['year'], y=comm_norm[comm],
                              name=comm.replace('_', ' ').title(),
                              line=dict(color=color, width=2)))

fig.add_vrect(x0=2019.5, x1=2020.5, fillcolor='rgba(255,0,0,0.1)',
               annotation_text='COVID', annotation_position='top left')
fig.add_vrect(x0=2021.5, x1=2022.5, fillcolor='rgba(255,100,0,0.1)',
               annotation_text='Energy\nCrisis', annotation_position='top left')
fig.update_layout(title='Commodity Prices Indexed (2010 = 100)',
                   yaxis_title='Index (2010=100)', height=400,
                   template='plotly_dark')
fig.show()

In [ ]:
# Merge commodity YoY changes with economic data
comm_yoy = comm_df.copy()
for col in ['oil_brent', 'gold', 'copper', 'wheat']:
    comm_yoy[f'{col}_yoy'] = comm_yoy[col].pct_change() * 100

merged = econ_df.merge(comm_yoy[['year', 'oil_brent_yoy', 'gold_yoy', 'copper_yoy', 'wheat_yoy']],
                        on='year', how='left')

corr_cols = ['gdp_growth_pct', 'inflation_pct', 'unemployment_pct',
              'oil_brent_yoy', 'gold_yoy', 'copper_yoy', 'wheat_yoy']
corr_matrix = merged[corr_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
             center=0, vmin=-1, vmax=1, ax=ax,
             linewidths=0.5, square=True)
ax.set_title('Commodity × Macro Correlation Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

---
## 6. Feature Correlation Heatmap

In [ ]:
numeric_cols = ['gdp_growth_pct', 'inflation_pct', 'unemployment_pct',
                 'interest_rate_pct', 'govt_debt_pct_gdp', 'gdp_per_capita',
                 'current_account_pct_gdp', 'oil_dependency_score']

corr = econ_df[numeric_cols].corr().round(3)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
    colorscale='RdYlGn', zmid=0, zmin=-1, zmax=1,
    text=corr.values.round(2),
    texttemplate='%{text}',
    colorbar=dict(title='Correlation'),
))
fig.update_layout(title='Economic Indicator Correlation Matrix',
                   height=500, template='plotly_dark')
fig.show()

print('\nStrongest positive correlations with GDP Growth:')
print(corr['gdp_growth_pct'].sort_values(ascending=False).iloc[1:4])
print('\nStrongest negative correlations with GDP Growth:')
print(corr['gdp_growth_pct'].sort_values().iloc[:3])

---
## 7. Country Clustering (K-Means)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Use 2022 snapshot for clustering
cluster_features = ['gdp_growth_pct', 'inflation_pct', 'unemployment_pct',
                      'interest_rate_pct', 'govt_debt_pct_gdp', 'current_account_pct_gdp']

cluster_df = econ_df[econ_df['year'] == 2022][['iso3', 'country', 'region'] + cluster_features].dropna()
X = StandardScaler().fit_transform(cluster_df[cluster_features])

# Elbow curve
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, marker='o', color='#4caf8a', linewidth=2)
ax.set_title('Elbow Curve — Optimal K for Country Clustering')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.axvline(x=4, color='#e05c5c', linestyle='--', alpha=0.6, label='K=4 selected')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Apply K=4 clustering
km = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_df['cluster'] = km.fit_predict(X)

# PCA for 2D visualisation
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X)
cluster_df['pc1'] = pca_coords[:, 0]
cluster_df['pc2'] = pca_coords[:, 1]

CLUSTER_NAMES = {0: 'Stable Advanced', 1: 'High Volatility', 2: 'Emerging Growth', 3: 'Distressed'}
cluster_df['cluster_name'] = cluster_df['cluster'].map(CLUSTER_NAMES)

fig = px.scatter(cluster_df, x='pc1', y='pc2', color='cluster_name',
                  hover_name='country', size_max=12,
                  labels={'pc1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)',
                          'pc2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)',
                          'cluster_name': 'Cluster'},
                  title='Country Economic Clusters — 2022 (K-Means, PCA projection)',
                  template='plotly_dark')
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))
fig.update_layout(height=450)
fig.show()

print('\nCluster membership:')
for name, group in cluster_df.groupby('cluster_name'):
    print(f'  {name}: {list(group["iso3"].values)}')

---
## 8. Recession Signal Analysis

In [ ]:
# Define recession = 2 consecutive quarters of negative GDP growth (annual proxy: growth < 0)
recession_df = econ_df.copy()
recession_df['in_recession'] = (recession_df['gdp_growth_pct'] < 0).astype(int)

# Recession frequency per country
rec_freq = recession_df.groupby('country')['in_recession'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e05c5c' if v >= 3 else '#f0a050' if v >= 1 else '#4caf8a' for v in rec_freq.values]
ax.bar(rec_freq.index, rec_freq.values, color=colors)
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Years in Recession (2010–2023)')
ax.set_title('Recession Frequency by Country')
plt.tight_layout()
plt.show()

# Key recession predictors
print('\nRecession predictor correlations:')
recession_df['lead_recession'] = recession_df.groupby('iso3')['in_recession'].shift(-1)
pred_corr = recession_df[['gdp_growth_pct', 'inflation_pct', 'unemployment_pct',
                            'interest_rate_pct', 'govt_debt_pct_gdp', 'lead_recession']]
print(pred_corr.corr()['lead_recession'].sort_values())

---
## 9. XGBoost Feature Importance

In [ ]:
# Load from model_metrics.json (computed in ml/forecasting.py)
print('=== GDP Growth Model ===')
print(f"MAE: {model_metrics['gdp_model']['mae']:.3f}pp")
print(f"R²:  {model_metrics['gdp_model']['r2']:.3f}")
print(f"Top features: {model_metrics['gdp_model']['top_features']}")
print()
print('=== Inflation Model ===')
print(f"MAE: {model_metrics['inflation_model']['mae']:.3f}pp")
print(f"R²:  {model_metrics['inflation_model']['r2']:.3f}")
print(f"Top features: {model_metrics['inflation_model']['top_features']}")

In [ ]:
# Quick re-train for feature importance visualisation
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, r2_score

econ_sorted = econ_df.sort_values(['iso3', 'year'])

for col in ['gdp_growth_pct', 'inflation_pct', 'unemployment_pct', 'interest_rate_pct']:
    for lag in [1, 2, 3]:
        econ_sorted[f'{col}_lag{lag}'] = econ_sorted.groupby('iso3')[col].shift(lag)

econ_sorted = econ_sorted.dropna()
econ_sorted = pd.get_dummies(econ_sorted, columns=['region'], prefix='region')

FEATURES = [c for c in econ_sorted.columns
             if c.endswith(('_lag1', '_lag2', '_lag3')) or c.startswith('region_')
             or c in ['gdp_per_capita', 'population']]

X = econ_sorted[FEATURES].fillna(0)
y = econ_sorted['gdp_growth_pct']

X_train, X_test = X[econ_sorted['year'] <= 2020], X[econ_sorted['year'] > 2020]
y_train, y_test = y[econ_sorted['year'] <= 2020], y[econ_sorted['year'] > 2020]

model = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f'Test MAE: {mean_absolute_error(y_test, preds):.3f}pp')
print(f'Test R²:  {r2_score(y_test, preds):.3f}')

importance_df = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_})
importance_df = importance_df.nlargest(15, 'importance')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='#4a7fb5')
ax.set_title('XGBoost Feature Importance — GDP Growth Forecasting', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## Summary

| Finding | Detail |
|---------|--------|
| COVID impact | 2020 caused the sharpest single-year GDP contraction across all 30 countries |
| Inflation surge | 2022 energy crisis drove inflation to 10–15% across Europe, highest since the 1980s |
| Top GDP predictor | Lagged GDP growth (lag-1) is the single strongest predictor in the XGBoost model |
| Most vulnerable | High trade-concentration countries (e.g. SGP, BEL, NLD) are most exposed to bilateral shocks |
| Recession signals | Rising unemployment and interest rate hikes are the strongest forward-looking recession predictors |

---
*Next: Run `streamlit run dashboard/app.py` to explore the full interactive dashboard.*